<a href="https://colab.research.google.com/github/KalekyeRaychelle/Movie-Recommender/blob/9-label-movies-using-llm/movieLabelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.6 MB/s eta 0:00:00


In [3]:
from groq import Groq
client=Groq(api_key=groqAPIkey)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role":"user","content":"Say hello!"}
    ]
)
print(response.choices[0].message.content)

Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [4]:
models = client.models.list()
for model in models.data:
    print(model.id)

meta-llama/llama-prompt-guard-2-86m
allam-2-7b
meta-llama/llama-4-scout-17b-16e-instruct
moonshotai/kimi-k2-instruct-0905
qwen/qwen3-32b
whisper-large-v3-turbo
groq/compound-mini
moonshotai/kimi-k2-instruct
canopylabs/orpheus-arabic-saudi
llama-3.3-70b-versatile
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
whisper-large-v3
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
llama-3.1-8b-instant
groq/compound


In [6]:
import pandas as pd
movies=pd.read_csv('cleanedMovies.csv')
movies.head(30)

,movieId,title,genres,tmdbId,tags,description
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,"pixar, fun","Led by Woody, Andy's toys live happily in his ..."
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,"fantasy, magic board game, Robin Williams, game",When siblings Judy and Peter discover an encha...
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,"moldy, old",A family wedding reignites the ancient feud be...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,NaN,"Cheated on, mistreated and stepped on, the wom..."
4,5,Father of the Bride Part II (1995),Comedy,11862.0,"pregnancy, remake",Just when George Banks has recovered from his ...
5,6,Heat (1995),Action|Crime|Thriller,949.0,NaN,Obsessive master thief Neil McCauley leads a t...
6,7,Sabrina (1995),Comedy|Romance,11860.0,remake,"After her return from school in Paris, a playb..."
7,8,Tom and Huck (1995),Adventure|Children,45325.0,NaN,"A mischievous young boy, Tom Sawyer, witnesses..."
8,9,Sudden Death (1995),Action,9091.0,NaN,When a man's daughter is suddenly taken during...
9,10,GoldenEye (1995),Action|Adventure|Thriller,710.0,NaN,When a powerful satellite system falls into th...


### CLEAN GENRES

In [7]:
row=movies.iloc[0]
cleanGenres=row['genres'].replace('|',', ')
print(cleanGenres)

Adventure, Animation, Children, Comedy, Fantasy


In [8]:
intents = [
    "I want to cry",
    "I want to laugh",
    "I want to escape reality",
    "I want action",
    "I want drama",
    "I want to feel hopeful",
    "I want to feel happy",
    "I want to be scared",
    "I want to think deeply",
    "I want a love story",
    "I want a heartbreaking romance",
    "I want a coming of age story"
]

In [25]:
def labelMovie(title,description,tags,genres):
  prompt=f"""You are a movie expert. Given a movie's details, assign it exactly one intent from the list below.
  Only respond with exact intent text, nothing else. No explanation, no punctuation, just the intent.
  Intents:
  {chr(10).join(intents)}

  Movie Title: {title}
  Description: {description}
  Genres: {genres}
  Tags: {tags}

  Respond in this exact format:
  Intent: <the intent>
  Reason: <one sentence explaining why>
  Which single intent best describes what this movie delivers to the viewer?"""

  response=client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[{"role":"user","content": prompt}]
  )
  return response.choices[0].message.content.strip()

In [19]:
row=movies.iloc[259]
cleanGenres=row['genres'].replace('|',', ')
result=labelMovie(row['title'],row['description'],row['tags'],cleanGenres)
print(result)

Intent: I want drama
Reason: The movie Priest is classified as a drama and its description suggests a thought-provoking and emotionally challenging story that explores the priest's convictions and the events within his congregation.


In [27]:
row=movies.iloc[9604]
print(row)
cleanGenres=row['genres'].replace('|',', ')
result=labelMovie(row['title'],row['description'],row['tags'],cleanGenres)
print(result)

movieId                                                   190209
title                         Jeff Ross Roasts the Border (2017)
genres                                                    Comedy
tmdbId                                                  487541.0
tags                                                         NaN
description    Roastmaster Jeff Ross explores the world surro...
Name: 9604, dtype: object
Intent: I want to laugh
Reason: The movie is a comedy that features Jeff Ross roasting various groups and individuals, indicating its primary intent is to entertain and amuse the audience.


In [11]:
print(movies.iloc[[1,5,10,50,100]][['title','genres','tags']])

                              title                      genres  \
1                    Jumanji (1995)  Adventure|Children|Fantasy   
5                       Heat (1995)       Action|Crime|Thriller   
10   American President, The (1995)        Comedy|Drama|Romance   
50                   Georgia (1995)                       Drama   
100         Before and After (1996)               Drama|Mystery   

                                                tags  
1    fantasy, magic board game, Robin Williams, game  
5                                                NaN  
10                               politics, president  
50                                               NaN  
100                                              NaN  


In [23]:
print(movies.iloc[[28,199,350,600,700,1900]][['title','genres','tags']])

                                                  title  \
28    City of Lost Children, The (Cité des enfants p...   
199                                      Exotica (1994)   
350   Highlander III: The Sorcerer (a.k.a. Highlande...   
600                                 Stupids, The (1996)   
700                            Wizard of Oz, The (1939)   
1900                                      Meteor (1979)   

                                      genres           tags  
28    Adventure|Drama|Fantasy|Mystery|Sci-Fi     kidnapping  
199                                    Drama            NaN  
350                           Action|Fantasy            NaN  
600                                   Comedy            NaN  
700       Adventure|Children|Fantasy|Musical  Dorothy, Toto  
1900                                  Sci-Fi            NaN  


In [21]:
for i in [1,5,10,50,100]:
  row=movies.iloc[i]
  cleanGenres=row['genres'].replace('|',', ')
  result=labelMovie(row['title'],row['description'],row['tags'],cleanGenres)
  print(result)

Intent: I want to escape reality
Reason: The movie's fantasy setting and magical elements allow viewers to temporarily escape into a world of adventure and imagination.
Intent: I want action
Reason: The movie's focus on daring heists and a cat-and-mouse game between a thief and a detective suggests a fast-paced and thrilling experience.
Intent: I want a love story
Reason: The movie focuses on the romantic relationship between the U.S. president and a lobbyist, blending comedy, drama, and romance genres.
Intent: I want drama
Reason: The movie Georgia is classified as a drama and its description focuses on the complex and troubled relationship between two sisters, indicating a dramatic storyline.
Intent: I want drama
Reason: The movie's genres and plot, which revolves around a family dealing with a serious accusation and its repercussions, suggest a dramatic and serious tone.


In [24]:
for i in [28,199,350,600,700,1900]:
  row=movies.iloc[i]
  cleanGenres=row['genres'].replace('|',',')
  results=labelMovie(row['title'],row['description'],row['tags'],cleanGenres)
  print(result)

Intent: I want drama
Reason: The movie's genres and plot, which revolves around a family dealing with a serious accusation and its repercussions, suggest a dramatic and serious tone.
Intent: I want drama
Reason: The movie's genres and plot, which revolves around a family dealing with a serious accusation and its repercussions, suggest a dramatic and serious tone.
Intent: I want drama
Reason: The movie's genres and plot, which revolves around a family dealing with a serious accusation and its repercussions, suggest a dramatic and serious tone.
Intent: I want drama
Reason: The movie's genres and plot, which revolves around a family dealing with a serious accusation and its repercussions, suggest a dramatic and serious tone.
Intent: I want drama
Reason: The movie's genres and plot, which revolves around a family dealing with a serious accusation and its repercussions, suggest a dramatic and serious tone.
Intent: I want drama
Reason: The movie's genres and plot, which revolves around a fam

In [28]:
from tqdm import tqdm

In [31]:
sample=movies.tail(1000).copy()
intentsList=[]
reasonsList=[]
for _,row in tqdm(sample.iterrows(),total=len(sample)):
  cleanGenres=row['genres'].replace('|',',')
  result=labelMovie(row['title'],row['description'],row['tags'],cleanGenres)

  try:
    intent=result.split('Intent:')[1].split('Reason: ')[0].strip()
    reason=result.split('Reason:')[1].strip()
  except:
    intent=result
    reason=None

  intentsList.append(intent)
  reasonsList.append(reason)

sample['intent']=intentsList
sample['reason']=reasonsList
sample['title','intent','reason'].head (10)

 28%|██▊       | 281/1000 [10:30<26:52,  2.24s/it]  


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kkf4h5f4ebe91bdm3ae9ernk` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 244. Please try again in 3m30.816s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [34]:

print(len(intentsList))
print(len(reasonsList))

281
281


In [38]:
partialLabels= sample.iloc[:281].copy()
partialLabels['intent']=intentsList
partialLabels['reason']=reasonsList
partialLabels.to_csv('partialLabels.csv',index=False)


## USING FACEBOOK/BART-LARGE-MNLI

In [39]:
from transformers import pipeline

classifier=pipeline("zero-shot-classification",model="facebook/bart-large-mnli")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [40]:
row=movies.iloc[0]
text=f"{row['title']}. {row['description']}. Genres: {row['genres'].replace('|',',')}. Tags: {row['tags']}"

In [41]:
result=classifier(text,intents)
print(f"Movie:{ row['title']}")
print(f"Intent: {result['labels'][0]}")
print(f"Score: {result['scores'][0]:.2f}")

Movie:Toy Story (1995)
Intent: I want to laugh
Score: 0.32
